# pyskills API
> API details

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fastcore.docments import MarkdownRenderer
from importlib.metadata import entry_points
from inspect import signature
import importlib.util, types, inspect, collections

### Overview

A plugin system allowing Python packages to register "skills" — units of LLM-usable functionality — via standard Python entry points. An LLM host (e.g. solveit) discovers available skills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen skills into context.

### Entry Point Convention

Packages register skills under the group `pyskills`:

```toml
[project.entry-points.pyskills]
my_skill = "mypackage.skill"
```

The value is a module path (no `:attribute` needed). The module's docstring first paragraph serves as the skill description.

### Skill Module Contract

A skill module MUST have:

- **Docstring** — first paragraph is the short description shown to the LLM for skill selection; remaining will be read by the LLM to get full details on the skill.

A skill module MAY have:

- **`__all__`** — the available symbols imported.

### Discovery API

```python
def list_pyskills() -> dict[str, str]
```
Returns `{name: description}` for all registered skills, using `find_spec` + AST parsing — no imports.

```python
import mypackage.skill
```
Standard python native import

```python
mypackage.skill.__doc__
```

Standard python native docs.

### Host Integration

The host (e.g. solveit, claude code, codex, ...) would:

1. Call `list_pyskills()` at startup to build a skill catalogue
2. Include the list with each prompt
3. Call `import {module}` followed by `doc({module})` for chosen skills

## Listing pyskills

In [ ]:
ep = entry_points()
es = first(ep.select(group='pyskills', name='pyskills.skill'))
es

EntryPoint(name='pyskills.skill', value='pyskills.skill', group='pyskills')

In [ ]:
#| export
def ep_desc(ep):
    "First paragraph of docstring for entry point `ep`, without importing it"
    spec = importlib.util.find_spec(ep.value.split(':')[0])
    if not spec or not spec.origin: return None
    tree = ast.parse(Path(spec.origin).read_text())
    doc = ast.get_docstring(tree)
    if not doc: return None
    return doc.split('\n\n')[0]

In [ ]:
print(ep_desc(es))

Pyskills is a plugin system allowing Python packages to register "skills" — units of LLM-usable functionality — via standard Python entry points. An LLM host (e.g. solveit) discovers available skills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen skills into context using standard imports. To get the details: `import pyskills.skill; pyskills.skill.__doc__`


In [ ]:
#| export
def list_pyskills():
    "Returns {module: description} for all pyskills. To load a module, use `import {module}` then view `doc({module})"
    return {ep.value: ep_desc(ep) for ep in entry_points().select(group='pyskills')}

In [ ]:
list_pyskills()

{'pyskills.skill': 'Pyskills is a plugin system allowing Python packages to register "skills" — units of LLM-usable functionality — via standard Python entry points. An LLM host (e.g. solveit) discovers available skills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen skills into context using standard imports. To get the details: `import pyskills.skill; pyskills.skill.__doc__`'}

## `allow`

In [ ]:
#| export
__pytools__ = collections.defaultdict(set)

In [ ]:
#| export
_all_ = ['__pytools__']

In [ ]:
#| export
def allow(*c, write_policy=None):
    def _wrap(v):
        if write_policy is None or v is ... or isinstance(v, tuple): return v
        return (v, write_policy)
    for o in c:
        if isinstance(o, dict):
            for k,v in o.items(): __pytools__[k].update(_wrap(x) for x in listify(v))
        else:
            mod = sys.modules.get(getattr(o, '__module__', '__main__'), sys.modules.get('__main__'))
            __pytools__[mod].add(_wrap(o.__name__))
    if len(c)==1 and callable(c[0]): return c[0]

`__pytools__` is a `defaultdict(set)` mapping classes/modules to their allowed method/function names (or `...` for all public methods). Values can be plain strings or `(name, WritePolicy)` tuples for write-checked methods. `allow` registers entries — callables are added under their module, dicts go directly into `__pytools__`.

In [ ]:
def _test_fn(): pass
_test_fn.__module__ = '__main__'
_test_fn.__name__ = 'my_test_func'
allow(_test_fn)
assert 'my_test_func' in __pytools__[sys.modules['__main__']]
allow({str: ['zfill']})
assert 'zfill' in __pytools__[str]
allow({list: ...})
assert ... in __pytools__[list]
__pytools__[sys.modules['__main__']].discard('my_test_func')


### Write policies

In [ ]:
#| export
def chk_dest(p, ok_dests):
    resolved = str(Path(p).resolve())
    if not any(resolved == (rd := str(Path(d).expanduser().resolve())) or resolved.startswith(rd + '/') for d in ok_dests):
        raise PermissionError(f"Write to '{p}' not allowed; permitted: {ok_dests}")

`chk_dest` resolves a path and verifies it falls under one of the allowed destination prefixes. Raises `PermissionError` if not. Used by all `WritePolicy` subclasses.

In [ ]:
chk_dest('/tmp/foo.txt', ['/tmp'])
try: chk_dest('/etc/passwd', ['/tmp'])
except PermissionError: print("Correctly blocked /etc/passwd")

Correctly blocked /etc/passwd


In [ ]:
#| export
class WritePolicy:
    "Base for write destination policies"
    def __call__(self, obj, args, kwargs, ok_dests): raise NotImplementedError

class PosWritePolicy(WritePolicy):
    "Check positional/keyword arg is an allowed write destination"
    def __init__(self, pos=0, kw=None): store_attr()
    def __call__(self, obj, args, kwargs, ok_dests):
        p = kwargs.get(self.kw) if self.kw and self.kw in kwargs else args[self.pos] if self.pos < len(args) else None
        if p is not None: chk_dest(p, ok_dests)

class PathWritePolicy(WritePolicy):
    "Check resolved Path self, optionally also target args"
    def __init__(self, target_pos=None, target_kw=None): store_attr()
    def __call__(self, obj, args, kwargs, ok_dests):
        chk_dest(obj, ok_dests)
        if self.target_pos is not None and self.target_pos < len(args): chk_dest(args[self.target_pos], ok_dests)
        if self.target_kw and self.target_kw in kwargs: chk_dest(kwargs[self.target_kw], ok_dests)

class OpenWritePolicy(WritePolicy):
    "Check open() only when mode is writable"
    def __call__(self, obj, args, kwargs, ok_dests):
        mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
        if any(c in mode for c in 'wax+'): chk_dest(args[0] if args else kwargs.get('file'), ok_dests)

Three `WritePolicy` subclasses handle different write-checking patterns.

In [ ]:
pp = PosWritePolicy(1, 'dst')
pp(None, ['src', '/tmp/ok'], {}, ['/tmp'])
try: pp(None, ['src', '/root/bad'], {}, ['/tmp'])
except PermissionError: print("PosWritePolicy blocked /root/bad")

pwp = PathWritePolicy()
pwp(Path('/tmp/f.txt'), [], {}, ['/tmp'])
try: pwp(Path('/etc/f.txt'), [], {}, ['/tmp'])
except PermissionError: print("PathWritePolicy blocked /etc/f.txt")

owp = OpenWritePolicy()
owp(None, ['/tmp/f.txt', 'w'], {}, ['/tmp'])
owp(None, ['/etc/passwd', 'r'], {}, ['/tmp'])
try: owp(None, ['/root/f.txt', 'w'], {}, ['/tmp'])
except PermissionError: print("OpenWritePolicy blocked write to /root/f.txt")

PosWritePolicy blocked /root/bad
PathWritePolicy blocked /etc/f.txt
OpenWritePolicy blocked write to /root/f.txt


Write policies are stored as `(name, WritePolicy)` tuples directly inside `__pytools__` sets.

## doc

In [ ]:
#| export
def doc(sym)->str:
    "Docs for `sym`. modules: classes/functions with 1st docstring line; classes: method sigs; functions: sig+docments"
    if isinstance(sym, type): return _doc_class(sym)
    if isinstance(sym, types.ModuleType): return _doc_module(sym)
    return str(MarkdownRenderer(sym))

def _fmt_method(name, method, prefix=''):
    try: sig = str(signature(method))
    except (ValueError, TypeError): sig = '(...)'
    d = getattr(method, '__doc__', None)
    res = f'    {prefix}def {name}{sig}: ...'
    if d and name != '__init__': res += f'  # {d.splitlines()[0].strip()}'
    return res

In [ ]:
import pyskills.skill

In [ ]:
print(doc(pyskills.skill.f))

def f(
    x:int=0, # the input
)->str: # the output
"""A test function"""


In [ ]:
#| export
def _doc_class(sym):
    bases = ','.join(b.__name__ for b in sym.__mro__[1:-1]) if sym.__mro__[1:-1] else ''
    parts = [f'class {sym.__name__}({bases}):' if bases else f'class {sym.__name__}:']
    init = getattr(sym, '__init__', None)
    if init and init is not object.__init__: parts.append(_fmt_method('__init__', init))
    for name,raw in sorted(sym.__dict__.items()):
        if name.startswith('_'): continue
        if isinstance(raw, property): parts.append(_fmt_method(name, raw.fget, '@property\n    '))
        elif isinstance(raw, classmethod): parts.append(_fmt_method(name, raw.__func__, '@classmethod\n    '))
        elif isinstance(raw, staticmethod): parts.append(_fmt_method(name, raw.__func__, '@staticmethod\n    '))
        elif callable(raw): parts.append(_fmt_method(name, raw))
    d = sym.__doc__ or (init and init.__doc__)
    if d: parts.insert(1, '    """' + inspect.cleandoc(d).replace('\n', '\n    ') + '"""')
    return parts[0] + '\n' + '\n'.join(parts[1:])

In [ ]:
print(doc(pyskills.skill.F))

class F(str):
    """Some class.
    More info about it."""
    def __init__(self): ...
    def f(self, x: int = 0) -> str: ...  # A test method
    @property
    def g(self) -> str: ...  # A test prop


In [ ]:
#| export
def _doc_module(mod):
    parts = [f'module {mod.__name__}:']
    if mod.__doc__: parts.append('"""' + inspect.cleandoc(mod.__doc__) + '"""\n')
    for name in sorted(dir(mod)):
        if name.startswith('_'): continue
        obj = getattr(mod, name, None)
        if obj is None or getattr(obj, '__module__', None) != mod.__name__: continue
        ds = getattr(obj, '__doc__', None)
        if not ds and isinstance(obj, type): ds = getattr(getattr(obj, '__init__', None), '__doc__', None)
        d = (ds or '').splitlines()
        comment = f': ...  # {d[0].strip()}' if d and d[0].strip() else ''
        if isinstance(obj, type):
            bases = ','.join(b.__name__ for b in obj.__mro__[1:-1])
            base_str = f'({bases})' if bases else ''
            parts.append(f'class {name}{base_str}{comment}')
        elif callable(obj):
            try: sig = str(signature(obj))
            except (ValueError, TypeError): sig = '(...)'
            parts.append(f'def {name}{sig}{comment}')
    return '\n'.join(parts)

In [ ]:
print(doc(pyskills.skill))

module pyskills.skill:
"""Pyskills is a plugin system allowing Python packages to register "skills" — units of LLM-usable functionality — via standard Python entry points. An LLM host (e.g. solveit) discovers available skills without importing them, reads lightweight descriptions via AST inspection, and selectively loads chosen skills into context using standard imports. To get the details: `import pyskills.skill; pyskills.skill.__doc__`

This is where we'll add more details about it."""

class F(str): ...  # Some class.
def f(x: int = 0) -> str: ...  # A test function


## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()